# Notebook 17 — Excel Automation: xlsxwriter and openpyxl

Demonstrates the two-library pattern used in Phase 1:
`xlsxwriter` for writing formatted workbooks with formula strings,
`openpyxl` for reading them back and verifying content.

**Uses:** `xlsxwriter`, `openpyxl`, `ibnr_utils.teaching_export.export_to_excel`  
**Produces:** Example workbooks in `data/output/teaching_phase1/`,
             openpyxl read-back verification results

## Step 1 — Configure Python path

**Purpose:** Locate the project root and add `src/` to `sys.path`.  
**Uses:** `pathlib`, `sys.path`  
**Produces:** `project_root`, `tmp_dir` for scratch workbooks

In [1]:
import pathlib, sys, tempfile

_cwd = pathlib.Path.cwd()
project_root = _cwd
# Walk up from cwd until we find the directory that contains both
# src/ibnr_utils/ and README.md — that is the project root regardless of
# where the notebook kernel was started.


# for _ in range(5):
#     if (project_root / 'src' / 'ibnr_utils').exists() and (project_root / 'README.md').exists():
#         break
#     project_root = project_root.parent

src_path = project_root / 'src'
# Add src/ to sys.path so Python can import ibnr_utils without installing the package.
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# All outputs for this notebook land inside teaching_phase1/.
OUT_DIR = project_root / 'data' / 'output' / 'teaching_phase1'
print(f'project_root: {project_root}')
print(f'OUT_DIR     : {OUT_DIR}')


project_root: c:\Users\derec\Desktop\Diplomado\diplomado-ml-Dereck\Modulo5\m5t2_sesion2
OUT_DIR     : c:\Users\derec\Desktop\Diplomado\diplomado-ml-Dereck\Modulo5\m5t2_sesion2\data\output\teaching_phase1


## Step 2 — xlsxwriter basics: write a minimal workbook

**Purpose:** Demonstrate the xlsxwriter API: `Workbook`, `add_worksheet`, `add_format`,
`write`, `write_number`, `write_formula`, `write_blank`, `close`.  
**Produces:** `scratch_basics.xlsx` in `OUT_DIR/validation/`  
**How it works:**  
- `wb = xlsxwriter.Workbook(path)` → `ws = wb.add_worksheet(name)` is the minimal pattern.
- `add_format({...})` creates reusable cell formats (bold, background, number mask).
- `write_number` stores a float; `write_formula` stores a formula **string** with a cached value.
- `wb.close()` finalises the file — always call it, or use a context manager.
- xlsxwriter can only *write* — it cannot read existing files.
- All row/col indices are **0-based**: `ws.write(0, 0, ...)` is cell A1.

In [3]:
!pip install xlsxwriter

  Using cached xlsxwriter-3.2.9-py3-none-any.whl.metadata (2.7 kB)
Using cached xlsxwriter-3.2.9-py3-none-any.whl (175 kB)


In [5]:
import xlsxwriter

# Scratch workbook lives in validation/ so it does not mix with
# the reference and formula-calculation workbooks in teaching_phase1/.
scratch_path = OUT_DIR / 'validation' / 'scratch_basics.xlsx'

# xlsxwriter creates a new file — it cannot open or edit an existing one.
wb = xlsxwriter.Workbook(str(scratch_path))
# Add a single worksheet called "Demo".  The workbook can hold many sheets.
ws = wb.add_worksheet('Demo')

# Formats are reusable style objects attached to the workbook (not the sheet).
hdr_fmt  = wb.add_format({'bold': True, 'bg_color': '#D9E1F2'})   # bold blue header
num_fmt  = wb.add_format({'num_format': '#,##0.00'})               # 2 dp with comma
pct_fmt  = wb.add_format({'num_format': '0.0%'})                   # percentage
note_fmt = wb.add_format({'italic': True, 'font_color': '#7030A0', 'text_wrap': True})

# Column widths: set_column(first_col, last_col, width) — col indices are 0-based.
# Column 0 (A) gets width 20; columns 1-3 (B-D) each get width 16.
ws.set_column(0, 0, 20)
ws.set_column(1, 3, 16)

# Headers — row 0 in Python = Excel row 1.
# write(row, col, value, format): both row and col are 0-based integers.
# write(0, 0, ...) → cell A1;  write(0, 1, ...) → B1, and so on.
for j, hdr in enumerate(['accident_period', 'current_incurred', 'atu', 'ultimate']):
    ws.write(0, j, hdr, hdr_fmt)

# Data rows: Python rows 1-4 = Excel rows 2-5.
data = [
    ('2021-01', 100_000.0, 1.08, None),
    ('2021-06', 200_000.0, 1.15, None),
    ('2022-01', 150_000.0, 1.22, None),
    ('2022-06',  80_000.0, 1.35, None),
]
for i, (ap, actual, atu_val, _) in enumerate(data):
    r = i + 1   # Python row index (0-based); Excel row number = r+1 (1-based)
    ws.write(r, 0, ap)                        # column A (index 0): accident_period
    ws.write_number(r, 1, actual, num_fmt)    # column B (index 1): current_incurred
    ws.write_number(r, 2, atu_val, num_fmt)   # column C (index 2): ATU factor
    # write_formula(row, col, formula_string, format, cached_value)
    # Formula strings use Excel's 1-based row numbers: Python r=1 → Excel row 2.
    # First data row: Demo!D2 = =B2*C2  (current_incurred × ATU = ultimate estimate).
    # cached_value is what openpyxl data_only=True reads back before Excel recalculates;
    # supplying actual*atu_val ensures the pre-recalc value is already correct.
    ws.write_formula(r, 3, f'=B{r+1}*C{r+1}', num_fmt, actual * atu_val)

ws.write(6, 0, 'Formula in column D: =B*C  (cached; press Ctrl+Alt+F9 to force Excel recalc)', note_fmt)

# close() finalises the ZIP-based .xlsx file; the file is unusable without it.
wb.close()
print(f'Written: {scratch_path.name} ({scratch_path.stat().st_size:,} bytes)')


Written: scratch_basics.xlsx (5,779 bytes)


## Step 3 — openpyxl read-back: inspect cell values and formulas

**Purpose:** Read the workbook just written with openpyxl and verify formula strings.  
**Uses:** `openpyxl.load_workbook`, cell `.value`, `.data_type`  
**Produces:** Printed cell contents for verification  
**How it works:**  
- `openpyxl.load_workbook(path)` opens the file for reading.
- By default, `data_only=False` → formula cells return the formula string (`=B2*C2`).
- `data_only=True` → returns the **cached value** Excel last stored (requires the file to have been saved by Excel with recalculation).
- Because xlsxwriter wrote the file (not Excel), the cached values are our Python-computed numbers.
- When `data_only=True` and the file was never opened in Excel, openpyxl returns the xlsxwriter-supplied cached value. This is why we supply it explicitly: `write_formula(r, c, formula, fmt, cached_value)`.

In [6]:
import openpyxl

# data_only=False (the default): formula cells return the formula string, e.g. '=B2*C2'.
# This lets us verify that formulas were written correctly.
wb_r = openpyxl.load_workbook(scratch_path, data_only=False)
ws_r = wb_r['Demo']
print('Formulas (data_only=False):')
for row in ws_r.iter_rows(min_row=2, max_row=5, values_only=False):
    ap, actual, atu_cell, ult_cell = row
    print(f'  {ap.value:<12}  actual={actual.value:>12,.0f}  atu={atu_cell.value}  ult_formula={ult_cell.value!r}')
wb_r.close()

print()

# data_only=True: formula cells return the cached value stored alongside the formula.
# xlsxwriter wrote actual*atu_val as the cached value, so reading back with
# data_only=True returns those Python-computed numbers unchanged.
# After opening and saving in Excel, Excel replaces the cached value with its
# own recalculation — both should agree to within floating-point tolerance (~1e-8).
wb_d = openpyxl.load_workbook(scratch_path, data_only=True)
ws_d = wb_d['Demo']
print('Cached values (data_only=True):')
for row in ws_d.iter_rows(min_row=2, max_row=5, values_only=True):
    ap, actual, atu_val, ult_cached = row
    expected = actual * atu_val if actual and atu_val else None
    diff = abs(ult_cached - expected) if (ult_cached and expected) else None
    print(f'  {ap:<12}  cached_ult={ult_cached:>12,.2f}  expected={expected:>12,.2f}  diff={diff:.2e}')
wb_d.close()


Formulas (data_only=False):
  2021-01       actual=     100,000  atu=1.08  ult_formula='=B2*C2'
  2021-06       actual=     200,000  atu=1.15  ult_formula='=B3*C3'
  2022-01       actual=     150,000  atu=1.22  ult_formula='=B4*C4'
  2022-06       actual=      80,000  atu=1.35  ult_formula='=B5*C5'

Cached values (data_only=True):
  2021-01       cached_ult=  108,000.00  expected=  108,000.00  diff=0.00e+00
  2021-06       cached_ult=  230,000.00  expected=  230,000.00  diff=0.00e+00
  2022-01       cached_ult=  183,000.00  expected=  183,000.00  diff=0.00e+00
  2022-06       cached_ult=  108,000.00  expected=  108,000.00  diff=0.00e+00


## Step 4 — Column index helper: `_xl_col`

**Purpose:** Demonstrate the `_xl_col` helper that converts 0-based column indices to
Excel letter notation.  
**Uses:** `ibnr_utils.teaching_export._xl_col` (internal helper, tested here for pedagogy)  
**How it works:** xlsxwriter writes cells by (row, col) integers, but formula strings
need Excel letter notation (A, B, … Z, AA, AB, …). For large triangles with 84 development
columns, manually tracking letters is error-prone. `_xl_col(0)` → "A", `_xl_col(25)` → "Z",
`_xl_col(26)` → "AA". The function uses a base-26 encoding with 1-indexing for the carry.

In [7]:
# from ibnr_utils.teaching_export import _xl_col

# _xl_col converts a 0-based column index to Excel's letter notation.
# xlsxwriter writes cells by (row, col) integer pairs, but formula strings
# must use Excel letter notation — '=B2*C2' instead of integer coordinates.
# For a triangle with 84 development columns the mapping reaches two letters:
# _xl_col(0) → 'A', _xl_col(25) → 'Z', _xl_col(26) → 'AA', _xl_col(83) → 'CF'.
checks = [(0,'A'), (1,'B'), (25,'Z'), (26,'AA'), (27,'AB'), (51,'AZ'), (52,'BA'), (83,'CF')]
print('_xl_col examples:')
for idx, expected in checks:
    result = _xl_col(idx)
    status = 'OK' if result == expected else f'FAIL (got {result!r})'
    print(f'  _xl_col({idx:3d}) = {result!r:<6}  expected {expected!r}  {status}')


_xl_col examples:


NameError: name '_xl_col' is not defined

## Step 5 — Export the teaching workbook via `export_to_excel`

**Purpose:** Run the full teaching export and write the standard 6-sheet workbook.  
**Uses:** `TeachingExportConfig`, `run_teaching_export`, `export_to_excel`  
**Produces:** `teaching_development_method_2022-12.xlsx`  
**How it works:** `export_to_excel` calls `_write_workbook` (xlsxwriter) to build the
6-sheet reference workbook. The Formula Calculations sheet's formulas reference the Python
Reference sheet. To experiment with formula changes, save a copy under a new name before editing.

In [9]:
# from ibnr_utils.teaching_export import TeachingExportConfig, run_teaching_export, export_to_excel

CSV_PATH = project_root / 'data' / 'output' / 'synthetic_monthly_multiconcept_database.csv'
CUTOFF   = '2022-12'

# Run the full Development Method pipeline: load CSV → apply cutoff →
# compute ATA/ATU/ultimates → return a TeachingExportResult that holds all
# matrices and the ultimate_table that the workbook writers consume.
# config = TeachingExportConfig(cutoff=CUTOFF, csv_path=CSV_PATH)
# result = run_teaching_export(config)

# Write demo copy to validation/ so the reference workbook in teaching_phase1/
# is never overwritten by this notebook.
xlsx_path = OUT_DIR / 'validation' / f'teaching_development_method_{CUTOFF}_demo.xlsx'

# export_to_excel writes the 6-sheet reference workbook with xlsxwriter.
# export_to_excel(result, xlsx_path)
# print(f'Workbook : {xlsx_path.name} — {xlsx_path.stat().st_size/1024:.1f} KB')


## Step 6 — Inspect all 6 sheets with openpyxl

**Purpose:** Read the reference workbook and report sheet names, row counts, and
a sample of cell values from each sheet.  
**Uses:** `openpyxl.load_workbook`  
**How it works:** Reading back a workbook that xlsxwriter wrote is the standard way to
verify its contents programmatically. The six sheets serve distinct purposes:
- **README**: metadata and assumptions (text cells).
- **Cumulative / Incremental**: triangle values (numeric cells, blanks for missing).
- **Factors**: ATA and ATU matrices (numeric cells).
- **Python Reference**: ultimate table — values calculated in Python; Formula Calculations formulas reference it.
- **Formula Calculations**: formula-driven columns linked to Python Reference.

In [11]:
import openpyxl

# Re-open the reference workbook we just wrote.
# xlsxwriter has no read capability, so openpyxl is the standard read-back tool.
wb = openpyxl.load_workbook(xlsx_path)
print(f'Sheet names: {wb.sheetnames}')
print()

for sname in wb.sheetnames:
    ws = wb[sname]
    # Count non-empty rows (at least one non-None cell in the row).
    n_rows = sum(1 for row in ws.iter_rows() if any(c.value is not None for c in row))
    n_cols = ws.max_column
    first_vals = [c.value for c in next(ws.iter_rows()) if c.value is not None][:4]
    print(f'  {sname:<18} rows={n_rows:<4} cols={n_cols:<4}  first row sample: {first_vals}')

wb.close()


Sheet names: ['README', 'Cumulative', 'Incremental', 'Factors', 'Python Reference', 'Formula Calculations']

  README             rows=17   cols=2     first row sample: ['Workbook type', 'Development Method — Reference Values']
  Cumulative         rows=85   cols=85    first row sample: ['accident_period', 0, 1, 2]
  Incremental        rows=85   cols=85    first row sample: ['accident_period', 0, 1, 2]
  Factors            rows=17   cols=84    first row sample: ['ATA Matrix (Age-to-Age Factors)']
  Python Reference   rows=86   cols=11    first row sample: ['Python Reference — values calculated in Python']
  Formula Calculations rows=86   cols=8     first row sample: ['Formula Calculations — formulas link to Python Reference.  Libraries write formulas but do not calculate them.  Press Ctrl+Alt+F9 in Excel and verify column E matches Python Reference Ultimate.']


## Step 7 — Verify formula strings in the Formula Calculations sheet

**Purpose:** Confirm that the Formula Calculations sheet contains formula strings referencing the
Python Reference sheet, not hard-coded values.  
**Uses:** `openpyxl.load_workbook(data_only=False)`, cell `.value`  
**How it works:** Checking for formula strings (values starting with `=`) rather than
hard-coded numbers is a programmatic way to enforce the formula structure. If values are pasted over formulas, the check will fail and highlight the affected cells.
This is also how CI could verify that a template workbook has not been corrupted.

In [12]:
# data_only=False returns formula strings (e.g. "='Python Reference'!D3").
# If we used data_only=True here, formula cells would return cached numbers and
# we could not verify that the formulas reference Python Reference by name.
wb = openpyxl.load_workbook(xlsx_path, data_only=False)
ws = wb['Formula Calculations']

formula_cells = [
    (cell.row, cell.column, cell.coordinate, cell.value)
    for row in ws.iter_rows()
    for cell in row
    if isinstance(cell.value, str) and cell.value.startswith('=')
]

print(f'Formula cells in Formula Calculations: {len(formula_cells)}')
print()
print('First 10 formula cells:')
for row, col, coord, formula in formula_cells[:10]:
    print(f'  {coord:<6}  {formula}')

# The Formula Calculations sheet has two types of formulas.
# Columns B-D link directly to Python Reference — e.g. Formula Calculations!B3 = ='Python Reference'!D3.
#   This means Python Reference row 3 and Formula Calculations row 3 refer to the same accident period;
#   both sheets start data at Excel row 3 (row 1 = banner, row 2 = column headers).
# Columns E-G derive the actuarial quantities from the linked values:
#   E = B*C  (current_incurred × ATU = projected ultimate estimate)
#   F = B/D  (current_incurred / pct_developed = alternate ultimate via percent-developed)
#   G = E-B  (projected ultimate − current_incurred = additional development)
ref_formulas = [f for *_, f in formula_cells if "'Python Reference'" in f or 'Python Reference' in f]
print(f'\nFormulas referencing Python Reference sheet: {len(ref_formulas)}')

wb.close()


Formula cells in Formula Calculations: 504

First 10 formula cells:
  B3      ='Python Reference'!D3
  C3      ='Python Reference'!E3
  D3      ='Python Reference'!F3
  E3      =B3*C3
  F3      =B3/D3
  G3      =E3-B3
  B4      ='Python Reference'!D4
  C4      ='Python Reference'!E4
  D4      ='Python Reference'!F4
  E4      =B4*C4

Formulas referencing Python Reference sheet: 252


## Step 8 — Formula vs cached value audit

**Purpose:** Compare xlsxwriter-cached values against Python-computed ultimates.  
**Uses:** `openpyxl` (data_only=True), `result.ultimate_table`  
**Produces:** Max absolute difference (should be 0.0 since Excel hasn't recalculated yet)  
**How it works:** When xlsxwriter writes `write_formula(r, c, formula, fmt, cached)`,
it stores `cached` as the cell value that Excel sees before any recalculation.
Since we supply `actual * atu` as the cached value, `data_only=True` returns our
Python results. After pressing Ctrl+Alt+F9 in Excel, Excel recalculates
the formulas and the values may diverge slightly from Python due to floating-point
differences in Excel's arithmetic — typically within 1e-8.

In [13]:
import pandas as pd

# data_only=True returns cached values — the numbers xlsxwriter stored alongside
# each formula when it wrote the file.  These are the Python-computed values, not
# the result of Excel evaluating the formula.  A diff of 0.0 here confirms that
# xlsxwriter stored exactly the values we computed in Python.
wb = openpyxl.load_workbook(xlsx_path, data_only=True)
ws = wb['Formula Calculations']

# Column E = Excel letter E = 0-based index 4 = row[4] in values_only mode.
# Excel row layout: row 1 = merged header banner, row 2 = column labels,
# data rows start at row 3.  Column E formula: =B*C = current_incurred × ATU.
excel_ultimates = []
for row in ws.iter_rows(min_row=3, values_only=True):  # row 1=header, row 2=header2, data from row 3
    if row[0] is None:
        break
    excel_ultimates.append(row[4])   # column E (index 4, 0-based) = projected ultimate
wb.close()

python_ultimates = result.ultimate_table['Ultimate'].dropna().tolist()

assert len(excel_ultimates) == len(python_ultimates), (
    f'Count mismatch: excel={len(excel_ultimates)} python={len(python_ultimates)}'
)
diffs = [abs(e - p) for e, p in zip(excel_ultimates, python_ultimates) if e is not None and p is not None]
print(f'Cached values compared : {len(diffs)}')
print(f'Max absolute difference: {max(diffs):.2e}')
print(f'Audit result           : {"OK — cached values match Python" if max(diffs) == 0 else "WARNING — nonzero diff"}')
print()
print('Windows check:')
print('  Save a copy of teaching_development_method_2022-12.xlsx (e.g. _working.xlsx).')
print('  Open the copy in Excel and press Ctrl+Alt+F9 to force full recalculation.')
print('  Column E (Formula Calculations sheet) should match column G (Python Reference sheet).')


NameError: name 'result' is not defined

## Summary

This notebook demonstrated:

1. **xlsxwriter write path**: `Workbook` → `add_worksheet` → `add_format` → `write_*` → `close`.
2. **Formula strings vs cached values**: `write_formula(row, col, "=B2*C2", fmt, cached)` stores
   both the formula and the Python-computed result. Excel uses the formula; openpyxl `data_only=True`
   reads the cached result.
3. **openpyxl read path**: `load_workbook(data_only=False)` for formula strings;
   `data_only=True` for cached values.
4. **Column helper**: `_xl_col(idx)` converts 0-based column index to Excel letter notation.
5. **Audit pattern**: compare openpyxl-read cached values against Python values to confirm
   the workbook was written correctly.
6. **Windows validation step**: save a copy of the workbook, open in Excel, press Ctrl+Alt+F9, verify Formula Calculations column E.

These patterns generalise to any project that writes actuarial triangle workbooks and needs
programmatic verification of their contents.